# Gold Layer — Yelp Businesses (Nested Flatten)

## Transformations
| Source field | Type | Gold output |
|---|---|---|
| `attributes` | `dict` (35 keys, một số là string-encoded Python dict) | 35+ flat columns |
| `hours` | `dict` (7 keys) | `hours_monday` … `hours_sunday` |
| `categories` | CSV string | `array<string>` |
| Còn lại | scalar | giữ nguyên |

Target table: `nessie.gold.businesses_flat`

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/`.
> Notebook chỉ setup SparkSession, gọi function, và làm phần interactive (monitor/validate).


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.spark_session import get_spark_session
from src import config

spark = get_spark_session("Yelp_Gold_Businesses")
print("✅ SparkSession ready")

✅ SparkSession ready


26/06/28 14:12:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## ⚠️ Bắt buộc — Ship `src/` package tới Spark executors

UDF (`to_bool_udf`, `parse_dict_udf`, `parse_categories_udf` trong `src/gold.py`) chạy trong **process Python riêng** trên executor, KHÔNG thừa hưởng `sys.path.append()` của notebook (driver).

→ Phải dùng `sparkContext.addPyFile()` để Spark tự ship code + thêm vào `sys.path` của worker.

Đóng gói lại với tên file **unique mỗi lần chạy** (timestamp) để tránh Spark cache file theo tên cũ khi `src/` vừa sửa code (tương tự gotcha "S3A JAR caching" đã gặp với named volume). Dùng `project_root` đã tính ở cell trên (không hardcode `/home/jovyan` — tránh phụ thuộc path container).

In [2]:
import shutil, time

_zip_base = str(project_root / f"src_{int(time.time())}")
shutil.make_archive(_zip_base, "zip", str(project_root), "src")
spark.sparkContext.addPyFile(f"{_zip_base}.zip")

print(f"✅ Đã ship {_zip_base}.zip tới Spark executors")

✅ Đã ship /home/iceberg/src_1782655944.zip tới Spark executors


## Step 1 — Đọc raw business.json vào Spark

In [3]:
raw_df = spark.read.json(config.BUSINESS_JSON_PATH)

print(f"Total businesses : {raw_df.count():,}")
print(f"Total columns    : {len(raw_df.columns)}")
print()
raw_df.printSchema()

Total businesses : 150,346
Total columns    : 14

root
 |-- address: string (nullable = true)
 |-- attributes: struct (nullable = true)
 |    |-- AcceptsInsurance: string (nullable = true)
 |    |-- AgesAllowed: string (nullable = true)
 |    |-- Alcohol: string (nullable = true)
 |    |-- Ambience: string (nullable = true)
 |    |-- BYOB: string (nullable = true)
 |    |-- BYOBCorkage: string (nullable = true)
 |    |-- BestNights: string (nullable = true)
 |    |-- BikeParking: string (nullable = true)
 |    |-- BusinessAcceptsBitcoin: string (nullable = true)
 |    |-- BusinessAcceptsCreditCards: string (nullable = true)
 |    |-- BusinessParking: string (nullable = true)
 |    |-- ByAppointmentOnly: string (nullable = true)
 |    |-- Caters: string (nullable = true)
 |    |-- CoatCheck: string (nullable = true)
 |    |-- Corkage: string (nullable = true)
 |    |-- DietaryRestrictions: string (nullable = true)
 |    |-- DogsAllowed: string (nullable = true)
 |    |-- DriveThru: stri

In [4]:
# Xem 1 sample record để confirm schema
import json
sample = raw_df.limit(1).toPandas().to_dict(orient="records")[0]
print(json.dumps(
    {k: str(v)[:120] for k, v in sample.items()},
    indent=2, ensure_ascii=False
))

{
  "address": "1616 Chapala St, Ste 2",
  "attributes": "Row(AcceptsInsurance=None, AgesAllowed=None, Alcohol=None, Ambience=None, BYOB=None, BYOBCorkage=None, BestNights=None, ",
  "business_id": "Pns2l4eNsfO8kk83dixA6A",
  "categories": "Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists",
  "city": "Santa Barbara",
  "hours": "None",
  "is_open": "0",
  "latitude": "34.4266787",
  "longitude": "-119.7111968",
  "name": "Abby Rappoport, LAC, CMQ",
  "postal_code": "93101",
  "review_count": "7",
  "stars": "5.0",
  "state": "CA"
}


## Step 2 & 3 — Flatten nested fields

UDFs (`safe_parse_python_dict`, `parse_categories`, `to_bool`) và toàn bộ logic flatten giờ nằm trong `src/gold.py` — xem `flatten_businesses()`.

In [5]:
from src.gold import flatten_businesses

flat_df = flatten_businesses(raw_df)
print(f"Columns sau flatten + expand: {len(flat_df.columns)}")
flat_df.printSchema()

Columns sau flatten + expand: 70
root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- is_open: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- attr_restaurants_takeout: boolean (nullable = true)
 |-- attr_restaurants_delivery: boolean (nullable = true)
 |-- attr_restaurants_reservations: boolean (nullable = true)
 |-- attr_outdoor_seating: boolean (nullable = true)
 |-- attr_wifi_raw: string (nullable = true)
 |-- attr_bike_parking: boolean (nullable = true)
 |-- attr_wheelchair_accessible: boolean (nullable = true)
 |-- attr_happy_hour: boolean (nullable = true)
 |-- attr_

## Step 4 — Ghi vào Gold Iceberg table

In [6]:
from src.gold import write_gold_businesses

print("[*] Đang ghi vào nessie.gold.businesses_flat...")
count = write_gold_businesses(flat_df, spark)
print(f"✅ Ghi xong! Total records: {count:,}")

[*] Đang ghi vào nessie.gold.businesses_flat...


✅ Ghi xong! Total records: 150,346


## Step 5 — Validation

In [7]:
# Validation 1: Schema tổng quan
from pyspark.sql.functions import col

print("=" * 55)
print("VALIDATION 1: TABLE OVERVIEW")
print("=" * 55)

gold = spark.table(config.TABLE_GOLD_BUSINESSES)
total = gold.count()
null_attrs = gold.filter(col("attr_restaurants_takeout").isNull()).count()
null_hours = gold.filter(col("hours_monday").isNull()).count()

print(f"  Total businesses       : {total:,}")
print(f"  Total columns          : {len(gold.columns)}")
print(f"  Null attr_takeout      : {null_attrs:,} ({null_attrs/total*100:.1f}%)")
print(f"  Null hours_monday      : {null_hours:,} ({null_hours/total*100:.1f}%)")

VALIDATION 1: TABLE OVERVIEW


  Total businesses       : 150,346
  Total columns          : 70
  Null attr_takeout      : 92,594 (61.6%)
  Null hours_monday      : 35,872 (23.9%)


In [8]:
# Validation 2: Categories được parse đúng
print("=" * 55)
print("VALIDATION 2: CATEGORIES PARSING")
print("=" * 55)

spark.sql(f"""
    SELECT name, categories, size(categories) as num_cats
    FROM {config.TABLE_GOLD_BUSINESSES}
    WHERE size(categories) > 3
    ORDER BY num_cats DESC
    LIMIT 5
""").show(truncate=False)

print("\nTop 10 categories:")
spark.sql(f"""
    SELECT cat, COUNT(*) as cnt
    FROM {config.TABLE_GOLD_BUSINESSES}
    LATERAL VIEW explode(categories) tmp AS cat
    GROUP BY cat
    ORDER BY cnt DESC
    LIMIT 10
""").show()

VALIDATION 2: CATEGORIES PARSING


+-----------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+
|name                                     |categories                                                                                                                                                                                                                                                                                                                                                                                                    

In [9]:
# Validation 3: Nested dict fields được flatten đúng
print("=" * 55)
print("VALIDATION 3: NESTED DICT FLATTEN")
print("=" * 55)

spark.sql(f"""
    SELECT
        name, attr_restaurants_takeout, attr_outdoor_seating,
        parking_garage, parking_street, ambience_casual, ambience_romantic,
        meal_lunch, meal_dinner, hours_monday, hours_friday
    FROM {config.TABLE_GOLD_BUSINESSES}
    WHERE parking_garage IS NOT NULL
      AND hours_monday IS NOT NULL
    LIMIT 5
""").show(truncate=False)

VALIDATION 3: NESTED DICT FLATTEN
+------------------+------------------------+--------------------+--------------+--------------+---------------+-----------------+----------+-----------+------------+------------+
|name              |attr_restaurants_takeout|attr_outdoor_seating|parking_garage|parking_street|ambience_casual|ambience_romantic|meal_lunch|meal_dinner|hours_monday|hours_friday|
+------------------+------------------------+--------------------+--------------+--------------+---------------+-----------------+----------+-----------+------------+------------+
|Target            |false                   |false               |false         |false         |NULL           |NULL             |NULL      |NULL       |8:0-22:0    |8:0-23:0    |
|St Honore Pastries|true                    |false               |false         |true          |NULL           |NULL             |NULL      |NULL       |7:0-20:0    |7:0-21:0    |
|Famous Footwear   |NULL                    |NULL                |

In [10]:
# Validation 4: Analytics query — chứng minh Gold layer dùng được
print("=" * 55)
print("VALIDATION 4: ANALYTICS QUERY")
print("=" * 55)

spark.sql(f"""
    SELECT
        state, COUNT(*) AS total_businesses, ROUND(AVG(stars), 2) AS avg_stars,
        SUM(CAST(attr_outdoor_seating AS INT)) AS with_outdoor_seating,
        SUM(CAST(attr_restaurants_takeout AS INT)) AS with_takeout
    FROM {config.TABLE_GOLD_BUSINESSES}
    WHERE is_open = 1
    GROUP BY state
    ORDER BY total_businesses DESC
    LIMIT 10
""").show()

print("\nBusiness 'all-inclusive' (có tất cả tiện ích):")
spark.sql(f"""
    SELECT name, city, state, stars, review_count
    FROM {config.TABLE_GOLD_BUSINESSES}
    WHERE attr_outdoor_seating = true
      AND attr_restaurants_delivery = true
      AND parking_lot = true
      AND is_open = 1
    ORDER BY stars DESC, review_count DESC
    LIMIT 10
""").show(truncate=False)

VALIDATION 4: ANALYTICS QUERY
+-----+----------------+---------+--------------------+------------+
|state|total_businesses|avg_stars|with_outdoor_seating|with_takeout|
+-----+----------------+---------+--------------------+------------+
|   PA|           26289|      3.6|                2814|        8619|
|   FL|           21540|     3.63|                2864|        6231|
|   TN|            9600|     3.58|                1323|        3083|
|   IN|            8946|      3.6|                1105|        3071|
|   MO|            8363|     3.57|                1219|        2832|
|   AZ|            8108|     3.62|                 930|        1869|
|   LA|            7676|      3.7|                1134|        2441|
|   NJ|            7031|     3.47|                 678|        2652|
|   NV|            6277|     3.77|                 492|        1175|
|   AB|            4346|     3.46|                 492|        1737|
+-----+----------------+---------+--------------------+------------+


Bu

In [11]:
# Iceberg snapshots — xem lịch sử ghi
print("=" * 55)
print("ICEBERG SNAPSHOTS — Gold Layer")
print("=" * 55)

spark.sql(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM {config.TABLE_GOLD_BUSINESSES}.snapshots
    ORDER BY committed_at
""").show(truncate=False)

ICEBERG SNAPSHOTS — Gold Layer
+-------------------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                                                                                     